# Exploratory Data Analysis

Describe the full prepared cohort for the Any Readmission project. Each row represents one eligible inpatient encounter selected for one patient. This notebook uses the full cohort for descriptive summaries.

## 1. Setup

Load the prepared CSV using its verified schema and manifest. Keep encounter and patient identifiers for integrity checks. Display aggregate context rather than individual records.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    directory
    for directory in (working_directory, *working_directory.parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.eda.data import load_eda_inputs

eda = load_eda_inputs(project_root=project_root)

## 2. Cohort Overview

The preparation manifest records the number of encounters and unique patients at each stage. Cleaning preserves all raw rows. Cohort selection excludes encounters with the configured death and hospice discharge codes, then keeps the smallest eligible numeric encounter ID for each patient.

In [ ]:
print(f"Prepared cohort: {len(eda.cohort):,} rows and {eda.cohort.shape[1]} columns.")
print(f"Retained encounters with unclear discharge descriptions: {eda.unclear_discharge_count:,}.")
display(eda.cohort_flow)

**Observation.** The final stage has one row per patient. The cohort also includes encounters with unclear discharge descriptions.

**Limitation.** The smallest eligible ID identifies a representative encounter. Encounter timestamps are unavailable for establishing whether it was the first admission.

## 3. Data Dictionary

Columns are listed in prepared CSV order. The period describes the information represented by the field. The original three-category outcome remains in the cohort. Field definitions draw on [Strack et al. (2014), Table 1](https://pmc.ncbi.nlm.nih.gov/articles/PMC3996476/#tab1).

In [ ]:
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(eda.column_dictionary)

## 4. Source Feature Decisions

The preparation audit identifies dropped source columns, replaced diagnosis codes, and derived fields that were considered but not created. Each reason is carried over from the approved preparation step.

In [ ]:
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(eda.source_feature_decisions)

## 5. Admission Code Descriptions

Admission Type and Admission Source use separate code lists. A code with an unclear description is recorded in the lookup. An unmapped code is absent from that lookup. The table shows the codes observed in the cohort.

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(eda.admission_descriptions.loc[eda.admission_descriptions["observed"]])

## 6. Data Completeness

An empty CSV field is counted separately from the source categories `None` in laboratory results and `Missing` or `Unmapped` in diagnosis groups. Admission codes with unclear descriptions are also reported separately. All percentages in this section use the full prepared cohort as the denominator.

In [ ]:
from src.eda.descriptive import profile_completeness, profile_numeric

completeness = profile_completeness(eda.cohort, eda.admission_descriptions)
print(
    f"{completeness.empty_field_count:,} empty fields across "
    f"{completeness.patients_with_empty_fields:,} patients."
)
display(completeness.missing_summary.round({"empty_percent": 4}))
display(completeness.missing_patterns.round({"percent": 4}))

In [ ]:
display(completeness.special_values.round({"percent": 4}))
display(completeness.lab_recording_patterns.round({"percent": 4}))

**Observation.** The empty fields occur in race and gender. The laboratory columns have no empty fields, although most patients have no recorded result for either test.

**Implication.** The proportion of empty fields describes the CSV structure. The laboratory and diagnosis tables describe which source information is available for interpretation.

**Limitation.** A result recorded as `None` does not establish whether the test was performed. An admission code with an unclear description remains an observed source code.

## 7. Numeric Distributions

The profiles cover all eight numeric predictors. Percentiles use linear interpolation among nonempty observations. The denominator for the zero percentage is the valid count. The plots show the full observed range.

In [ ]:
numeric = profile_numeric(eda.cohort)

with pd.option_context("display.max_columns", None):
    display(numeric.numeric_profile.round({"zero_percent_of_valid": 2}))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from src.eda.plots import plot_numeric_distributions

figures = plot_numeric_distributions(numeric.numeric_frequency, cohort_size=len(eda.cohort))
for figure in figures.values():
    plt.close(figure)
    display(figure)

### Prior Visits

The positive count for each service type is the denominator for its conditional percentiles. The second table counts the number of service types with at least one recorded visit for each patient. An empty group label represents an unknown count when a service field is missing.

In [ ]:
display(numeric.prior_visit_positive_profile)
display(numeric.prior_visit_coverage.round({"percent": 4}))

**Observation.** The median is zero for each prior service count. Among patients with positive counts, the values extend well above the median. As an example, the maximum outpatient count is above its positive-group 99th percentile.

**Implication.** The cohort-wide zero share and the positive-only distribution describe different parts of the same variable. The logarithmic count axis makes the rare high values visible.

**Limitation.** The visit counts describe recorded service use in the year before the representative encounter. High values are retained as observed. Their magnitude alone is insufficient evidence of a data error.